# Fine-tune: Regression

We fine-tune the pretrained KumoTabular small for the regression dataset (Allstate Claims Severity, target `loss`) loaded in notebook air_finetuning/01_data_preparation. 

**Compute:** AI Runtime (H100), environment version: AI v6

### Governance and lifecycle of the finetuned model

We recommend the **MLflow + UC Model Registry** approach for production use, as it provides out-of-the-box governance and lifecycle management. The Volume checkpoint approach is included as a simpler alternative if you want to get started before learning about MLflow.

1. **Volume checkpoint** — the model's `state_dict` is saved as a `.pt` file to a Unity Catalog Volume via `torch.save()` during the training loop. This is straightforward but requires manual tracking of run IDs, checkpoint paths, and model versions.

2. **MLflow + UC Model Registry** *(recommended)* — the model is logged with `mlflow.pytorch.log_model()` and registered directly in Unity Catalog. This gives you:
   * **Versioning** — every finetuning run creates a new model version automatically, with full lineage back to the training run, parameters, and metrics.
   * **Lifecycle aliases** — promote versions to `@champion` / `@challenger`; downstream notebooks reference the alias and pick up new versions without code changes.
   * **Access control** — UC-level permissions on the registered model, rather than volume-level permissions on a raw `.pt` file.
   * **Serving-ready** — if real-time inference is ever needed, a UC-registered PyTorch model can be deployed to a GPU-backed Model Serving endpoint directly.

## Configuration and libraries

In [0]:
%pip install git+https://github.com/NVIDIA/structured-data-models.git --quiet -q

In [0]:
dbutils.library.restartPython()

In [0]:
import importlib.util
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
COMMON_PATH = os.path.join(REPO_ROOT, "common")
CONFIG_PATH = os.path.join(COMMON_PATH, "config.py")

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"Could not find config.py at {CONFIG_PATH}")

spec = importlib.util.spec_from_file_location("project_config", CONFIG_PATH)
project_config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(project_config)

CATALOG = project_config.CATALOG
SCHEMA = "default"
VOLUME_NAME = "finetuned_regression_models"
TABLE_NAME_PREFIX = "finetuning_regression"
TRAIN_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE_NAME_PREFIX}_train"
TARGET_COL = "loss"
ID_COL = None  # Allstate dataset (OpenML 45046) has no ID column

current_user = spark.sql("SELECT current_user()").collect()[0][0]
MLFLOW_EXPERIMENT_NAME = f"/Users/{current_user}/tabular-fm-databricks-finetune"

In [0]:
import gc, time

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

import mlflow
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, TensorSpec
import torch
import torch.nn.functional as F
import sdm
import sdm.processing as sp

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME_NAME}")
volume_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print(f"Catalog/schema: {CATALOG}.{SCHEMA}")
print(f"Volume: {volume_path}")
print(f"MLflow experiment: {MLFLOW_EXPERIMENT_NAME}")

## Load and prepare training data

In [0]:
# Load training data from UC
train_pdf = spark.table(TRAIN_TABLE).toPandas()
train_fit_pdf = train_pdf.dropna(subset=[TARGET_COL])

# Feature columns (exclude ID and target)
all_feature_cols = [c for c in train_fit_pdf.columns if c not in (ID_COL, TARGET_COL)]
feature_cols = all_feature_cols

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Train: {len(train_fit_pdf):,} rows")
print(f"Features: {len(feature_cols)} of {len(all_feature_cols)} total")
print(f"Target '{TARGET_COL}': mean={train_fit_pdf[TARGET_COL].mean():.4f} std={train_fit_pdf[TARGET_COL].std():.4f}")
print(f"Device: {device}")

## Fine-tuning

This follows the same full-coverage fine-tuning loop used in 02a: each epoch shuffles the training rows, chunks them into context/query blocks, and runs one optimizer step per chunk. 

For regression, the model outputs quantiles `q001`–`q999`. We fine-tune against the full quantile grid using the mean pinball loss across all quantiles, which is proportional to CRPS. Validation tracks:

* `pinball_q100` for the 0.1 quantile
* `pinball_q900` for the 0.9 quantile
* `rmse_q500` and `mae_q500` for the median (`q500`)
* `crps` from the full 999-quantile prediction

Because regression returns 999 outputs per row, GPU memory is tighter than in classification. The epoch/step logic is unchanged from 02a; only the hyperparameters below are tuned down to avoid OOM.

In [0]:
# Delete any large GPU tensors from prior runs, this is useful in case you are running multiple fine-tuning runs and need to clean GPU memory between runs.
for name in list(globals()):
    obj = globals()[name]
    if isinstance(obj, (torch.Tensor,)) and obj.is_cuda:
        del globals()[name]

# Also target known heavy objects from the training cell
for var in ["model", "train_table", "val_table", "table", "optimizer", "scaler", "scheduler", "best_state", "perm", "ctx", "out", "preds", "y", "context", "query", "loss"]:
    if var in globals():
        del globals()[var]

gc.collect()
torch.cuda.empty_cache()

free, total = torch.cuda.mem_get_info()
print(f"GPU memory: {free / 2**30:.1f} GiB free / {total / 2**30:.1f} GiB total")

In [0]:
# ─��� Fine-tuning configuration ───────────────────────────────────────
FT_MODEL_SIZE = "small"
FT_LR = 1e-5
FT_WARMUP_PROPORTION = 0.1          # fraction of total steps for linear warmup
TRAIN_VAL_SPLIT = 0.5
FT_MAX_EPOCHS = 500
FT_PATIENCE = 50
FT_CONTEXT_SIZE = 8192
FT_QUERY_SIZE = 2048
FT_FEATURE_PERCENTAGE_TRAIN = 1
FT_NUM_ESTIMATORS_TRAIN = 1
FT_NUM_ESTIMATORS_VAL = 2
FT_EVAL_CONTEXT_SIZE = 16384
FT_MIN_DELTA = 1e-4
FT_WEIGHT_DECAY = 0.01
FT_SEED = 0

MLFLOW_RUN_NAME = "kumo_regression_finetuning_process"

QUANTILE_LEVELS = torch.arange(1, 1000, device=device, dtype=torch.float32) / 1000.0
Q100_IDX = 99
Q500_IDX = 499
Q900_IDX = 899

torch.manual_seed(FT_SEED)


def pinball_loss(preds: torch.Tensor, target: torch.Tensor, quantiles: torch.Tensor) -> torch.Tensor:
    if preds.ndim == 3:
        preds = preds.squeeze(0)
    if target.ndim == 1:
        target = target.unsqueeze(-1)
    quantiles = quantiles.to(device=preds.device, dtype=preds.dtype).view(1, -1)
    target = target.to(dtype=preds.dtype)
    error = target - preds
    return torch.maximum(quantiles * error, (quantiles - 1.0) * error)


def summarize_regression_metrics(preds: torch.Tensor, target: torch.Tensor) -> dict[str, float]:
    if preds.ndim == 3:
        preds = preds.squeeze(0)
    if target.ndim == 2:
        target = target.squeeze(-1)

    preds = preds.float()
    target = target.float()

    q100 = preds[:, Q100_IDX]
    q500 = preds[:, Q500_IDX]
    q900 = preds[:, Q900_IDX]

    pinball_grid = pinball_loss(preds, target.unsqueeze(-1), QUANTILE_LEVELS)
    crps = float((2.0 * pinball_grid.mean()).item())
    rmse = float(torch.sqrt(torch.mean((q500 - target) ** 2)).item())
    mae = float(torch.mean(torch.abs(q500 - target)).item())
    pinball_q100 = float(pinball_loss(q100.unsqueeze(-1), target.unsqueeze(-1), QUANTILE_LEVELS[Q100_IDX:Q100_IDX+1]).mean().item())
    pinball_q900 = float(pinball_loss(q900.unsqueeze(-1), target.unsqueeze(-1), QUANTILE_LEVELS[Q900_IDX:Q900_IDX+1]).mean().item())

    return {
        "pinball_q100": pinball_q100,
        "pinball_q900": pinball_q900,
        "rmse_q500": rmse,
        "mae_q500": mae,
        "crps": crps,
    }


def merge_quantile_estimators(preds: torch.Tensor) -> torch.Tensor:
    """Merge quantile predictions from multiple estimators by pooling and re-sorting.

    Each estimator outputs 999 quantiles (q001-q999). We pool all K*999 values
    per row, sort them, and pick 999 evenly-spaced quantiles from the sorted pool
    via linear interpolation. This produces a better-calibrated quantile function
    than pointwise averaging because it reconstructs the empirical CDF from all
    estimator samples before extracting quantiles.
    """
    if preds.ndim == 2:
        return preds
    if preds.shape[0] == 1:
        return preds.squeeze(0)
    n_est, n_rows, n_q = preds.shape
    # Pool all quantile values: (n_rows, K * 999)
    pooled = preds.permute(1, 0, 2).reshape(n_rows, n_est * n_q)
    pooled_sorted = pooled.sort(dim=1).values
    # Re-pick 999 quantiles via linear interpolation at the original quantile levels
    positions = QUANTILE_LEVELS * (n_est * n_q - 1)
    lower = positions.long()
    upper = (lower + 1).clamp(max=n_est * n_q - 1)
    frac = (positions - lower.float()).unsqueeze(0)  # (1, 999)
    return pooled_sorted[:, lower] * (1.0 - frac) + pooled_sorted[:, upper] * frac


# ── Prepare data ──────────────────────────────────────────────────
ft_cols = feature_cols + [TARGET_COL]
ft_df = train_fit_pdf[ft_cols].reset_index(drop=True)

stypes = sdm.infer_stypes(ft_df, overrides={TARGET_COL: "numerical"})
table = sdm.TableTensor.from_pandas(ft_df, stypes=stypes, device=device)

train_idx, val_idx = train_test_split(
    torch.arange(len(ft_df)).numpy(),
    test_size=TRAIN_VAL_SPLIT,
    random_state=FT_SEED,
)
train_table = table[torch.from_numpy(train_idx).to(device)]
val_table = table[torch.from_numpy(val_idx).to(device)]
del table
gc.collect()

# Full-coverage chunking: chunk data and extract number of steps per epoch
chunk_size = FT_CONTEXT_SIZE + FT_QUERY_SIZE
steps_per_epoch = max(1, len(train_idx) // chunk_size)

print(f"Train: {len(train_idx):,} rows | Val: {len(val_idx):,} rows")
print(f"Features: {len(feature_cols)} | Device: {device}")
print(f"Chunking: {len(train_idx):,} rows / {chunk_size} = {steps_per_epoch} steps/epoch (full coverage)")

# ── Model & recipe ────────────────────────────────────────────────
model = sdm.models.KumoTabular(
    task="regression", size=FT_MODEL_SIZE, device=device, pretrained=True,
)

train_recipe = model.default_recipe()
train_recipe.target = sp.StypeDispatch(
    numerical=sp.Standardize(),
    categorical=[
        sp.AlignCategories(),
        sp.ShuffleCategories(method="shift"),
    ],
)
train_recipe.output = sp.Identity()


def evaluate():
    """Evaluate full val set in one pass, random subsampled context each call."""
    model.eval()
    ctx_idx = torch.randperm(train_table.size(0), device=device)[:FT_EVAL_CONTEXT_SIZE]
    ctx = train_table[ctx_idx]
    with torch.no_grad():
        with torch.amp.autocast('cuda', dtype=torch.float16):
            out = model(
                x_context=ctx.drop_columns(TARGET_COL),
                y_context=ctx[TARGET_COL],
                x_query=val_table.drop_columns(TARGET_COL),
                recipe=train_recipe,
                num_estimators=FT_NUM_ESTIMATORS_VAL,
            )
    preds = merge_quantile_estimators(out.numerical)
    y = val_table[TARGET_COL].numerical.squeeze(-1)
    metrics = summarize_regression_metrics(preds, y)
    del out, preds, y, ctx
    torch.cuda.empty_cache()
    return metrics


# ── Training loop with MLflow ─────────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=FT_WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda')

total_steps = FT_MAX_EPOCHS * steps_per_epoch
warmup_steps = int(total_steps * FT_WARMUP_PROPORTION)

scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[
    torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1e-8, end_factor=1.0, total_iters=warmup_steps),
    torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps),
], milestones=[warmup_steps])
print(f"LR schedule: warmup {warmup_steps} steps, cosine decay over {total_steps} total steps")

with mlflow.start_run(run_name=MLFLOW_RUN_NAME):
    run_id = mlflow.active_run().info.run_id
    ckpt_path = f"{volume_path}/kumo_tabular_{FT_MODEL_SIZE}_regression_finetuned_{run_id}.pt"

    mlflow.log_params({
        "vendor": "kumo",
        "model_type": "KumoTabular",
        "model_size": FT_MODEL_SIZE,
        "task": "regression",
        "dataset": "allstate_claims_severity",
        "target_col": TARGET_COL,
        "lr": FT_LR,
        "max_epochs": FT_MAX_EPOCHS,
        "patience": FT_PATIENCE,
        "steps_per_epoch": steps_per_epoch,
        "chunk_size": chunk_size,
        "context_size": FT_CONTEXT_SIZE,
        "query_size": FT_QUERY_SIZE,
        "feature_percentage_train": FT_FEATURE_PERCENTAGE_TRAIN,
        "num_estimators_train": FT_NUM_ESTIMATORS_TRAIN,
        "num_estimators_val": FT_NUM_ESTIMATORS_VAL,
        "eval_context_size": FT_EVAL_CONTEXT_SIZE,
        "early_stop_metric": "crps",
        "min_delta": FT_MIN_DELTA,
        "weight_decay": FT_WEIGHT_DECAY,
        "warmup_proportion": FT_WARMUP_PROPORTION,
        "n_features": len(feature_cols),
        "n_train": len(train_idx),
        "n_val": len(val_idx),
        "seed": FT_SEED,
        "device": device,
    })

    val_metrics = evaluate()
    print(
        f"epoch=0/{FT_MAX_EPOCHS} "
        f"val_pinball_q100={val_metrics['pinball_q100']:.4f} "
        f"val_pinball_q900={val_metrics['pinball_q900']:.4f} "
        f"val_rmse_q500={val_metrics['rmse_q500']:.4f} "
        f"val_mae_q500={val_metrics['mae_q500']:.4f} "
        f"val_crps={val_metrics['crps']:.4f}"
    )
    mlflow.log_metrics({f"ft_per_epoch_val_{k}": v for k, v in val_metrics.items()}, step=0)

    best_val_metrics = val_metrics.copy()
    best_val_crps = val_metrics["crps"]
    epochs_without_improvement = 0
    torch.save(model.state_dict(), ckpt_path)

    gc.collect()
    torch.cuda.empty_cache()

    t_start = time.time()
    for epoch in range(1, FT_MAX_EPOCHS + 1):
        t_epoch = time.time()
        model.train()
        total_loss = 0.0

        perm = torch.randperm(train_table.size(0), device=device)
        for step in range(steps_per_epoch):
            start = step * chunk_size
            chunk_idx = perm[start : start + chunk_size]
            context = train_table[chunk_idx[:FT_CONTEXT_SIZE]]
            query = train_table[chunk_idx[FT_CONTEXT_SIZE:]]

            n_feat = len(feature_cols)
            n_keep = max(1, int(n_feat * FT_FEATURE_PERCENTAGE_TRAIN))
            feat_idx = torch.randperm(n_feat, device=device)[:n_keep].sort().values.tolist()
            drop_cols = [feature_cols[i] for i in range(n_feat) if i not in feat_idx] + [TARGET_COL]

            optimizer.zero_grad()
            with torch.amp.autocast('cuda', dtype=torch.float16):
                out = model(
                    x_context=context.drop_columns(drop_cols),
                    y_context=context[TARGET_COL],
                    x_query=query.drop_columns(drop_cols),
                    recipe=train_recipe,
                    num_estimators=FT_NUM_ESTIMATORS_TRAIN,
                )
                preds = merge_quantile_estimators(out.numerical)
                y = query[TARGET_COL].numerical.squeeze(-1)
                loss = pinball_loss(preds, y.unsqueeze(-1), QUANTILE_LEVELS).mean()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += float(loss.detach())
            del context, query, out, preds, y, loss
            torch.cuda.empty_cache()

        del perm
        avg_loss = total_loss / steps_per_epoch
        gc.collect()
        torch.cuda.empty_cache()

        val_metrics = evaluate()

        gc.collect()
        torch.cuda.empty_cache()

        epoch_time = time.time() - t_epoch
        print(
            f"epoch={epoch}/{FT_MAX_EPOCHS} "
            f"val_pinball_q100={val_metrics['pinball_q100']:.4f} "
            f"val_pinball_q900={val_metrics['pinball_q900']:.4f} "
            f"val_rmse_q500={val_metrics['rmse_q500']:.4f} "
            f"val_mae_q500={val_metrics['mae_q500']:.4f} "
            f"val_crps={val_metrics['crps']:.4f} "
            f"train_loss={avg_loss:.4f} "
            f"time={epoch_time:.1f}s"
        )
        mlflow.log_metrics({f"ft_per_epoch_val_{k}": v for k, v in val_metrics.items()}, step=epoch)
        mlflow.log_metric("ft_per_epoch_train_loss", avg_loss, step=epoch)

        if val_metrics["crps"] < best_val_crps - FT_MIN_DELTA:
            best_val_crps = val_metrics["crps"]
            best_val_metrics = val_metrics.copy()
            epochs_without_improvement = 0
            torch.save(model.state_dict(), ckpt_path)
            print(f"  → New best (crps={best_val_crps:.4f})! Saved checkpoint.")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= FT_PATIENCE:
                print(f"  → Early stopping after {FT_PATIENCE} epochs without improvement.")
                break

    ft_time = time.time() - t_start

    best_state = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(best_state)

    mlflow.log_metrics({
        "ft_best_val_pinball_q100": best_val_metrics["pinball_q100"],
        "ft_best_val_pinball_q900": best_val_metrics["pinball_q900"],
        "ft_best_val_rmse_q500": best_val_metrics["rmse_q500"],
        "ft_best_val_mae_q500": best_val_metrics["mae_q500"],
        "ft_best_val_crps": best_val_metrics["crps"],
        "ft_finetune_time_s": ft_time,
        "ft_epochs_completed": min(epoch, FT_MAX_EPOCHS),
    })
    mlflow.log_artifact(ckpt_path, artifact_path="checkpoints")

    signature = ModelSignature(
        inputs=Schema([TensorSpec(np.dtype(np.float32), (-1, len(feature_cols)), "features")]),
        outputs=Schema([TensorSpec(np.dtype(np.float32), (-1, 999), "quantiles")]),
    )
    mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path="model",
        signature=signature,
    )

print(f"\n✓ Fine-tuning complete (run {run_id})")
print(f"  Best val pinball q100: {best_val_metrics['pinball_q100']:.4f}")
print(f"  Best val pinball q900: {best_val_metrics['pinball_q900']:.4f}")
print(f"  Best val rmse q500:    {best_val_metrics['rmse_q500']:.4f}")
print(f"  Best val mae q500:     {best_val_metrics['mae_q500']:.4f}")
print(f"  Best val CRPS:         {best_val_metrics['crps']:.4f}")
print(f"  Time: {ft_time:.1f}s")
print(f"  Checkpoint: {ckpt_path}")

> **Note:** even though the validation metric used for early stopping improves during fine-tuning, this does not guarantee better performance on the held-out test set. Fine-tuning can overfit to the train/validation distribution, and the gain may not transfer. Whether fine-tuning actually paid off is only confirmed in the inference notebooks (`03a` / `03b`), where we compare the pretrained and fine-tuned models on data neither has seen.


## Registering the model in UC

Register the best fine-tuning run into Unity Catalog for governance and lifecycle. The cell below uses the MLflow model already logged by the training cell.

In [0]:
# Register the already-logged model from the training run into UC
registered_model_name = f"{CATALOG}.{SCHEMA}.kumo_tabular_regression"
model_uri = f"runs:/{run_id}/model"

result = mlflow.register_model(model_uri=model_uri, name=registered_model_name)

print(f"✓ Model registered: {registered_model_name}")
print(f"  Version: {result.version}")
print(f"  Source: {model_uri}")